<a href="https://colab.research.google.com/github/tsjannoun123-netizen/AI-for-Med.Diagnos.-Prediction-AAI_643O_O11_202610/blob/main/Week6_Foundadtion_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import timm
from PIL import Image
import torchvision.transforms as T
import warnings
warnings.filterwarnings('ignore')

# Configuration
class Config:
    batch_size = 32
    learning_rate = 1e-4
    num_epochs = 20
    image_size = 224
    num_classes = 5  # pneumonia, edema, cardiomegaly, atelectasis, consolidation
    metadata_dim = 10  # age, sex, ICU_stay, etc.

# Define target pathologies
PATHOLOGIES = ['Pneumonia', 'Edema', 'Cardiomegaly', 'Atelectasis', 'Consolidation']

In [29]:
class MIMICCXRDataset(Dataset):
    def __init__(self, split_file, image_dir, metadata_df, transform=None, is_train=True):
        self.split_df = pd.read_csv(split_file)
        self.image_dir = image_dir
        self.metadata_df = metadata_df
        self.transform = transform
        self.is_train = is_train

        # Standardize metadata
        self.scaler = StandardScaler()
        if is_train:
            self.metadata_scaled = self.scaler.fit_transform(
                metadata_df[['age', 'gender', 'icu_stay', 'view_position']].values
            )
        else:
            self.metadata_scaled = self.scaler.transform(
                metadata_df[['age', 'gender', 'icu_stay', 'view_position']].values
            )

    def __len__(self):
        return len(self.split_df)

    def __getitem__(self, idx):
        study_id = self.split_df.iloc[idx]['study_id']
        subject_id = self.split_df.iloc[idx]['subject_id']

        # Load image
        img_path = f"{self.image_dir}/{subject_id}/{study_id}.jpg"
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        # Get labels
        labels = self.split_df.iloc[idx][PATHOLOGIES].values.astype(np.float32)

        # Get metadata
        metadata = self.metadata_scaled[idx].astype(np.float32)

        return {
            'image': image,
            'metadata': torch.tensor(metadata),
            'labels': torch.tensor(labels),
            'study_id': study_id
        }

# Data transforms
train_transform = T.Compose([
    T.Resize((Config.image_size, Config.image_size)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize((Config.image_size, Config.image_size)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [30]:
class ConcatenationFusion(nn.Module):
    def __init__(self, num_classes=Config.num_classes, metadata_dim=Config.metadata_dim):
        super().__init__()

        # Image encoder (ResNet-50)
        self.image_encoder = timm.create_model('resnet50', pretrained=True, num_classes=0)
        image_feat_dim = 2048

        # Metadata encoder
        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(image_feat_dim + 64, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, metadata):
        # Extract image features
        image_features = self.image_encoder(image)

        # Extract metadata features
        metadata_features = self.metadata_encoder(metadata)

        # Concatenate and classify
        combined = torch.cat([image_features, metadata_features], dim=1)
        output = self.classifier(combined)

        return output

In [31]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, num_classes=Config.num_classes, metadata_dim=Config.metadata_dim):
        super().__init__()

        # Image encoder with patch features
        self.image_encoder = timm.create_model('resnet50', pretrained=True, num_classes=0)
        self.image_encoder.global_pool = nn.Identity()
        self.image_encoder.fc = nn.Identity()

        # ResNet feature map to patches
        self.patch_embed = nn.Conv2d(2048, 512, kernel_size=1)

        # Metadata encoder
        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512)  # Match image feature dimension
        )

        # Cross-attention layer
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=512, num_heads=8, batch_first=True
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, metadata):
        batch_size = image.size(0)

        # Extract image features
        image_features = self.image_encoder.forward_features(image)

        # Convert to patches (B, 2048, 7, 7) -> (B, 512, 7, 7) -> (B, 49, 512)
        patches = self.patch_embed(image_features)
        patches = patches.flatten(2).transpose(1, 2)  # (B, 49, 512)

        # Metadata as query
        metadata_query = self.metadata_encoder(metadata).unsqueeze(1)  # (B, 1, 512)

        # Cross-attention: metadata queries attend to image patches
        attended_features, _ = self.cross_attention(
            query=metadata_query,
            key=patches,
            value=patches
        )

        # Classify
        output = self.classifier(attended_features.squeeze(1))

        return output

In [32]:
class MedCLIPAdaptation(nn.Module):
    def __init__(self, num_classes=Config.num_classes, metadata_dim=Config.metadata_dim):
        super().__init__()

        # Load pre-trained MedCLIP (simplified representation)
        # In practice, you would load the actual MedCLIP model
        self.medclip = self._load_medclip_backbone()
        # Update feature_dim to match the output of the image encoder (ResNet50 output is 2048)
        self.feature_dim = 2048

        # Metadata encoder
        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.ReLU(),
            # Adjust output dimension to match the original design intent for metadata features
            nn.Linear(128, 512)
        )
        # Update metadata feature dimension
        self.metadata_feature_dim = 512

        # Calculate the input size for the classifier
        classifier_input_size = self.feature_dim + self.metadata_feature_dim
        print(f"Calculated classifier input size: {classifier_input_size}")


        # Fusion and classification head
        self.classifier = nn.Sequential(
            # Adjust input size to match the concatenated features (image_features + metadata_features)
            nn.Linear(classifier_input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )


    def _load_medclip_backbone(self):
        # Placeholder for actual MedCLIP loading
        # This would be replaced with actual MedCLIP implementation
        backbone = timm.create_model('resnet50', pretrained=True, num_classes=0)
        backbone.fc = nn.Identity()
        return backbone

    def forward(self, image, metadata):
        # Extract image features using MedCLIP
        image_features = self.medclip(image)

        # Extract metadata features
        metadata_features = self.metadata_encoder(metadata)

        # Combine features
        combined = torch.cat([image_features, metadata_features], dim=1)
        output = self.classifier(combined)

        return output

# Partial Fine-tuning (updating only MLP parameters)
class PartialFineTuning(MedCLIPAdaptation):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Freeze MedCLIP backbone
        for param in self.medclip.parameters():
            param.requires_grad = False
        # Only classifier and metadata encoder are trainable

# Linear Probing
class LinearProbing(MedCLIPAdaptation):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Freeze entire MedCLIP backbone and metadata encoder
        for param in self.medclip.parameters():
            param.requires_grad = False
        for param in self.metadata_encoder.parameters():
            param.requires_grad = False
        # Only classifier head is trainable

In [33]:
# Run the comparative study
results = run_comparative_study()
print_comparison_table(results)

# Print detailed pathology-wise results for best model
best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
best_metrics = results[best_model_name]['metrics']

print(f"\nBest Model: {best_model_name}")
print("Pathology-wise Performance:")
for pathology, metrics in best_metrics['per_pathology'].items():
    print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

Calculated classifier input size: 2560
Calculated classifier input size: 2560
Calculated classifier input size: 2560

=== Training Concatenation Fusion ===
Trainable parameters: 24,732,165
Estimated GPU memory: 112.72 MB
Epoch 1/20
Train Loss: 0.6926
Val AUC: 0.5864
Val Acc: 0.5200
Val F1: 0.4294
---
Epoch 2/20
Train Loss: 0.6914
Val AUC: 0.5284
Val Acc: 0.5100
Val F1: 0.4133
---
Epoch 3/20
Train Loss: 0.6952
Val AUC: 0.5593
Val Acc: 0.4800
Val F1: 0.3740
---
Epoch 4/20
Train Loss: 0.6944
Val AUC: 0.4563
Val Acc: 0.4500
Val F1: 0.3375
---
Epoch 5/20
Train Loss: 0.6936
Val AUC: 0.5370
Val Acc: 0.4600
Val F1: 0.3330
---
Epoch 6/20
Train Loss: 0.6927
Val AUC: 0.4144
Val Acc: 0.4900
Val F1: 0.4086
---
Epoch 7/20
Train Loss: 0.6935
Val AUC: 0.4952
Val Acc: 0.5100
Val F1: 0.3575
---
Epoch 8/20
Train Loss: 0.6934
Val AUC: 0.3958
Val Acc: 0.3700
Val F1: 0.1837
---
Epoch 9/20
Train Loss: 0.6923
Val AUC: 0.5577
Val Acc: 0.5600
Val F1: 0.2995
---
Epoch 10/20
Train Loss: 0.6958
Val AUC: 0.4935
Val

In [34]:
# Run the comparative study
results = run_comparative_study()
print_comparison_table(results)

# Print detailed pathology-wise results for best model
best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
best_metrics = results[best_model_name]['metrics']

print(f"\nBest Model: {best_model_name}")
print("Pathology-wise Performance:")
for pathology, metrics in best_metrics['per_pathology'].items():
    print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

Calculated classifier input size: 2560
Calculated classifier input size: 2560
Calculated classifier input size: 2560

=== Training Concatenation Fusion ===
Trainable parameters: 24,732,165
Estimated GPU memory: 112.72 MB
Epoch 1/20
Train Loss: 0.6944
Val AUC: 0.5179
Val Acc: 0.4300
Val F1: 0.1120
---
Epoch 2/20
Train Loss: 0.6946
Val AUC: 0.4984
Val Acc: 0.4600
Val F1: 0.1231
---
Epoch 3/20
Train Loss: 0.6924
Val AUC: 0.5508
Val Acc: 0.5300
Val F1: 0.1333
---
Epoch 4/20
Train Loss: 0.6952
Val AUC: 0.5963
Val Acc: 0.4300
Val F1: 0.1429
---
Epoch 5/20
Train Loss: 0.6952
Val AUC: 0.5589
Val Acc: 0.4800
Val F1: 0.1241
---
Epoch 6/20
Train Loss: 0.6932
Val AUC: 0.5021
Val Acc: 0.5600
Val F1: 0.1379
---
Epoch 7/20
Train Loss: 0.6931
Val AUC: 0.4660
Val Acc: 0.4400
Val F1: 0.1077
---
Epoch 8/20
Train Loss: 0.6939
Val AUC: 0.3696
Val Acc: 0.5100
Val F1: 0.1379
---
Epoch 9/20
Train Loss: 0.6930
Val AUC: 0.4560
Val Acc: 0.5800
Val F1: 0.1419
---
Epoch 10/20
Train Loss: 0.6929
Val AUC: 0.5339
Val

In [35]:
# Run the comparative study
results = run_comparative_study()
print_comparison_table(results)

# Print detailed pathology-wise results for best model
best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
best_metrics = results[best_model_name]['metrics']

print(f"\nBest Model: {best_model_name}")
print("Pathology-wise Performance:")
for pathology, metrics in best_metrics['per_pathology'].items():
    print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

Calculated classifier input size: 2560
Calculated classifier input size: 2560
Calculated classifier input size: 2560

=== Training Concatenation Fusion ===
Trainable parameters: 24,732,165
Estimated GPU memory: 112.72 MB
Epoch 1/20
Train Loss: 0.6946
Val AUC: 0.4737
Val Acc: 0.4700
Val F1: 0.4333
---
Epoch 2/20
Train Loss: 0.6915
Val AUC: 0.4819
Val Acc: 0.4400
Val F1: 0.3467
---
Epoch 3/20
Train Loss: 0.6931
Val AUC: 0.4944
Val Acc: 0.4800
Val F1: 0.4616
---
Epoch 4/20
Train Loss: 0.6929
Val AUC: 0.4828
Val Acc: 0.5100
Val F1: 0.4190
---
Epoch 5/20
Train Loss: 0.6924
Val AUC: 0.5237
Val Acc: 0.5400
Val F1: 0.4495
---
Epoch 6/20
Train Loss: 0.6911
Val AUC: 0.5218
Val Acc: 0.4600
Val F1: 0.3854
---
Epoch 7/20
Train Loss: 0.6904
Val AUC: 0.4214
Val Acc: 0.5000
Val F1: 0.4086
---
Epoch 8/20
Train Loss: 0.6956
Val AUC: 0.4284
Val Acc: 0.5100
Val F1: 0.3994
---
Epoch 9/20
Train Loss: 0.6932
Val AUC: 0.5972
Val Acc: 0.4100
Val F1: 0.3256
---
Epoch 10/20
Train Loss: 0.6923
Val AUC: 0.5368
Val

In [36]:
# Run the comparative study
results = run_comparative_study()
print_comparison_table(results)

# Print detailed pathology-wise results for best model
best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
best_metrics = results[best_model_name]['metrics']

print(f"\nBest Model: {best_model_name}")
print("Pathology-wise Performance:")
for pathology, metrics in best_metrics['per_pathology'].items():
    print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

Calculated classifier input size: 2560
Calculated classifier input size: 2560
Calculated classifier input size: 2560

=== Training Concatenation Fusion ===
Trainable parameters: 24,732,165
Estimated GPU memory: 112.72 MB
Epoch 1/20
Train Loss: 0.6951
Val AUC: 0.4560
Val Acc: 0.5400
Val F1: 0.4571
---
Epoch 2/20
Train Loss: 0.6942
Val AUC: 0.4189
Val Acc: 0.4400
Val F1: 0.4468
---
Epoch 3/20
Train Loss: 0.6924
Val AUC: 0.6076
Val Acc: 0.5200
Val F1: 0.4780
---
Epoch 4/20
Train Loss: 0.6927
Val AUC: 0.5482
Val Acc: 0.5100
Val F1: 0.4364
---
Epoch 5/20
Train Loss: 0.6948
Val AUC: 0.5568
Val Acc: 0.5400
Val F1: 0.3977
---
Epoch 6/20
Train Loss: 0.6947
Val AUC: 0.4679
Val Acc: 0.4900
Val F1: 0.3928
---
Epoch 7/20
Train Loss: 0.6926
Val AUC: 0.5141
Val Acc: 0.4900
Val F1: 0.3176
---
Epoch 8/20
Train Loss: 0.6940
Val AUC: 0.5102
Val Acc: 0.4900
Val F1: 0.3338
---
Epoch 9/20
Train Loss: 0.6931
Val AUC: 0.5262
Val Acc: 0.5200
Val F1: 0.4140
---
Epoch 10/20
Train Loss: 0.6942
Val AUC: 0.4999
Val

In [37]:
# Run the comparative study
results = run_comparative_study()
print_comparison_table(results)

# Print detailed pathology-wise results for best model
best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
best_metrics = results[best_model_name]['metrics']

print(f"\nBest Model: {best_model_name}")
print("Pathology-wise Performance:")
for pathology, metrics in best_metrics['per_pathology'].items():
    print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

Calculated classifier input size: 2560
Calculated classifier input size: 2560
Calculated classifier input size: 2560

=== Training Concatenation Fusion ===
Trainable parameters: 24,732,165
Estimated GPU memory: 112.72 MB
Epoch 1/20
Train Loss: 0.6965
Val AUC: 0.5413
Val Acc: 0.4500
Val F1: 0.2427
---
Epoch 2/20
Train Loss: 0.6929
Val AUC: 0.6525
Val Acc: 0.4900
Val F1: 0.1864
---
Epoch 3/20
Train Loss: 0.6932
Val AUC: 0.4785
Val Acc: 0.5100
Val F1: 0.0923
---
Epoch 4/20
Train Loss: 0.6946
Val AUC: 0.4497
Val Acc: 0.5900
Val F1: 0.1576
---
Epoch 5/20
Train Loss: 0.6937
Val AUC: 0.5302
Val Acc: 0.4300
Val F1: 0.1037
---
Epoch 6/20
Train Loss: 0.6933
Val AUC: 0.4822
Val Acc: 0.5400
Val F1: 0.1333
---
Epoch 7/20
Train Loss: 0.6929
Val AUC: 0.5024
Val Acc: 0.5200
Val F1: 0.1500
---
Epoch 8/20
Train Loss: 0.6933
Val AUC: 0.5784
Val Acc: 0.5000
Val F1: 0.1333
---
Epoch 9/20
Train Loss: 0.6925
Val AUC: 0.3625
Val Acc: 0.5200
Val F1: 0.2613
---
Epoch 10/20
Train Loss: 0.6930
Val AUC: 0.5731
Val

In [38]:
# Run the comparative study
results = run_comparative_study()
print_comparison_table(results)

# Print detailed pathology-wise results for best model
best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
best_metrics = results[best_model_name]['metrics']

print(f"\nBest Model: {best_model_name}")
print("Pathology-wise Performance:")
for pathology, metrics in best_metrics['per_pathology'].items():
    print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

Calculated classifier input size: 2560
Calculated classifier input size: 2560
Calculated classifier input size: 2560

=== Training Concatenation Fusion ===
Trainable parameters: 24,732,165
Estimated GPU memory: 112.72 MB
Epoch 1/20
Train Loss: 0.6949
Val AUC: 0.5147
Val Acc: 0.5700
Val F1: 0.4486
---
Epoch 2/20
Train Loss: 0.6933
Val AUC: 0.5329
Val Acc: 0.4100
Val F1: 0.3520
---
Epoch 3/20
Train Loss: 0.6916
Val AUC: 0.3957
Val Acc: 0.4800
Val F1: 0.3994
---
Epoch 4/20
Train Loss: 0.6936
Val AUC: 0.5199
Val Acc: 0.5300
Val F1: 0.4236
---
Epoch 5/20
Train Loss: 0.6949
Val AUC: 0.4678
Val Acc: 0.5700
Val F1: 0.4381
---
Epoch 6/20
Train Loss: 0.6932
Val AUC: 0.6711
Val Acc: 0.5100
Val F1: 0.4241
---
Epoch 7/20
Train Loss: 0.6930
Val AUC: 0.4862
Val Acc: 0.5600
Val F1: 0.4456
---
Epoch 8/20
Train Loss: 0.6928
Val AUC: 0.4541
Val Acc: 0.5100
Val F1: 0.4190
---
Epoch 9/20
Train Loss: 0.6919
Val AUC: 0.5833
Val Acc: 0.4200
Val F1: 0.3953
---
Epoch 10/20
Train Loss: 0.6953
Val AUC: 0.4743
Val

In [ ]:
# Run the comparative study
results = run_comparative_study()
print_comparison_table(results)

# Print detailed pathology-wise results for best model
best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
best_metrics = results[best_model_name]['metrics']

print(f"\nBest Model: {best_model_name}")
print("Pathology-wise Performance:")
for pathology, metrics in best_metrics['per_pathology'].items():
    print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

In [39]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=4):
        super().__init__()
        self.rank = rank
        self.lora_A = nn.Linear(in_dim, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_dim, bias=False)
        self.scaling = 1.0 / rank

        # Initialize
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.lora_B(self.lora_A(x)) * self.scaling

class LoRACrossAttention(nn.Module):
    def __init__(self, original_layer, rank=4):
        super().__init__()
        self.original_layer = original_layer
        # Assuming original_layer has embed_dim attribute
        self.lora_q = LoRALayer(original_layer.embed_dim, original_layer.embed_dim, rank)
        self.lora_v = LoRALayer(original_layer.embed_dim, original_layer.embed_dim, rank)

    def forward(self, query, key, value, **kwargs):
        # Original forward pass (without gradients for original layer)
        with torch.no_grad():
             original_output, attn_weights = self.original_layer(query, key, value, **kwargs)


        # LoRA adaptation
        lora_q_output = self.lora_q(query)
        lora_v_output = self.lora_v(value)

        # Simple adaptation (in practice, more sophisticated integration)
        # Combining original output and LoRA outputs
        adapted_output = original_output + lora_q_output + lora_v_output # Example of simple addition


        return adapted_output, attn_weights

class MedCLIPWithLoRA(MedCLIPAdaptation):
    def __init__(self, rank=4, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # Re-initialize classifier to ensure correct input dimension after super().__init__()
        classifier_input_size = self.feature_dim + self.metadata_feature_dim
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, Config.num_classes) # Use Config.num_classes directly
        )

        # Add LoRA to cross-attention layers if they exist
        # This is a simplified example - actual implementation depends on MedCLIP architecture
        # Check if medclip has a cross_attention module
        found_cross_attention = False
        for name, module in self.medclip.named_modules():
            if isinstance(module, nn.MultiheadAttention): # Check for MultiheadAttention as a proxy
                # Assuming the first MultiheadAttention is the one to adapt
                original_attention_layer = module
                # Need to find the parent module to replace the child
                parent_name = name.rsplit('.', 1)[0]
                parent_module = self.medclip
                if parent_name:
                    for sub_name in parent_name.split('.'):
                        parent_module = getattr(parent_module, sub_name)

                setattr(parent_module, name.split('.')[-1], LoRACrossAttention(original_attention_layer, rank))
                found_cross_attention = True
                break # Adapt only the first found cross-attention for simplicity


        # Freeze original parameters
        for param in self.medclip.parameters():
            param.requires_grad = False
        for param in self.metadata_encoder.parameters():
             param.requires_grad = False
        for param in self.classifier.parameters():
             param.requires_grad = True


        # Unfreeze LoRA parameters if cross_attention was found and adapted
        if found_cross_attention:
            for name, module in self.medclip.named_modules():
                 if isinstance(module, LoRACrossAttention):
                     for param in module.parameters():
                         param.requires_grad = True

In [40]:
class Trainer:
    def __init__(self, model, train_loader, val_loader, config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config

        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=config.learning_rate, weight_decay=1e-4
        )
        self.criterion = nn.BCEWithLogitsLoss()
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=config.num_epochs
        )

        self.best_auc = 0
        self.train_losses = []
        self.val_metrics = []

    def train_epoch(self):
        self.model.train()
        total_loss = 0

        for batch in self.train_loader:
            images = batch['image'].cuda()
            metadata = batch['metadata'].cuda()
            labels = batch['labels'].cuda()

            self.optimizer.zero_grad()

            outputs = self.model(images, metadata)
            loss = self.criterion(outputs, labels)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    def evaluate(self):
        self.model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in self.val_loader:
                images = batch['image'].cuda()
                metadata = batch['metadata'].cuda()
                labels = batch['labels'].cuda()

                outputs = self.model(images, metadata)
                preds = torch.sigmoid(outputs)

                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())

        all_preds = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()

        # Calculate metrics for each pathology
        metrics = {}
        for i, pathology in enumerate(PATHOLOGIES):
            try:
                auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
                pred_binary = (all_preds[:, i] > 0.5).astype(int)
                acc = accuracy_score(all_labels[:, i], pred_binary)
                f1 = f1_score(all_labels[:, i], pred_binary, zero_division=0)

                metrics[pathology] = {
                    'auc': auc,
                    'accuracy': acc,
                    'f1': f1
                }
            except:
                metrics[pathology] = {'auc': 0, 'accuracy': 0, 'f1': 0}

        # Average metrics
        avg_auc = np.mean([m['auc'] for m in metrics.values()])
        avg_acc = np.mean([m['accuracy'] for m in metrics.values()])
        avg_f1 = np.mean([m['f1'] for m in metrics.values()])

        return {
            'per_pathology': metrics,
            'average': {'auc': avg_auc, 'accuracy': avg_acc, 'f1': avg_f1}
        }

    def train(self):
        for epoch in range(self.config.num_epochs):
            train_loss = self.train_epoch()
            val_metrics = self.evaluate()

            self.train_losses.append(train_loss)
            self.val_metrics.append(val_metrics)

            print(f'Epoch {epoch+1}/{self.config.num_epochs}')
            print(f'Train Loss: {train_loss:.4f}')
            print(f'Val AUC: {val_metrics["average"]["auc"]:.4f}')
            print(f'Val Acc: {val_metrics["average"]["accuracy"]:.4f}')
            print(f'Val F1: {val_metrics["average"]["f1"]:.4f}')
            print('---')

            self.scheduler.step()

            # Save best model
            if val_metrics["average"]["auc"] > self.best_auc:
                self.best_auc = val_metrics["average"]["auc"]
                torch.save(self.model.state_dict(), 'best_model.pth')

        return self.val_metrics

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def estimate_gpu_memory(model, batch_size, image_size):
    # Rough estimation of GPU memory usage
    model_params = count_parameters(model) * 4 / (1024 ** 2)  # MB
    activation_memory = batch_size * image_size * image_size * 3 * 4 / (1024 ** 2)  # MB
    return model_params + activation_memory

In [41]:
def run_comparative_study():
    # This would be the main execution function
    config = Config()

    # Load datasets (placeholder - actual paths needed)
    # train_dataset = MIMICCXRDataset(...)
    # val_dataset = MIMICCXRDataset(...)
    # train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    # val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    # Placeholder for dummy dataset and dataloader creation
    # Replace this with your actual dataset loading
    class DummyDataset(Dataset):
        def __init__(self, num_samples=100):
            self.num_samples = num_samples
            self.image_size = Config.image_size
            self.metadata_dim = Config.metadata_dim
            self.num_classes = Config.num_classes

        def __len__(self):
            return self.num_samples

        def __getitem__(self, idx):
            image = torch.randn(3, self.image_size, self.image_size)
            metadata = torch.randn(self.metadata_dim)
            labels = torch.randint(0, 2, (self.num_classes,)).float()
            study_id = f'study_{idx}'
            return {'image': image, 'metadata': metadata, 'labels': labels, 'study_id': study_id}

    train_dataset = DummyDataset(num_samples=100)
    val_dataset = DummyDataset(num_samples=20)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    models = {
        'Concatenation Fusion': ConcatenationFusion().cuda(),
        'Cross-Attention Fusion': CrossAttentionFusion().cuda(),
        'Partial Fine-tuning': PartialFineTuning().cuda(),
        'Linear Probing': LinearProbing().cuda(),
        'LoRA Adaptation': MedCLIPWithLoRA().cuda()
    }

    results = {}

    for name, model in models.items():
        print(f"\n=== Training {name} ===")

        # Count parameters and estimate memory
        trainable_params = count_parameters(model)
        memory_estimate = estimate_gpu_memory(model, config.batch_size, config.image_size)

        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Estimated GPU memory: {memory_estimate:.2f} MB")

        # Train and evaluate
        trainer = Trainer(model, train_loader, val_loader, config)
        metrics = trainer.train()

        results[name] = {
            'metrics': metrics[-1],  # Final epoch metrics
            'trainable_params': trainable_params,
            'gpu_memory': memory_estimate,
            'best_auc': trainer.best_auc
        }

    return results

def print_comparison_table(results):
    print("\n" + "="*80)
    print("COMPARATIVE RESULTS")
    print("="*80)
    print(f"{'Model':<25} {'Avg AUC':<10} {'Avg Acc':<10} {'Avg F1':<10} {'Params':<12} {'GPU Mem (MB)':<12}")
    print("-"*80)

    for name, result in results.items():
        avg_metrics = result['metrics']['average']
        print(f"{name:<25} {avg_metrics['auc']:.4f}    {avg_metrics['accuracy']:.4f}    "
              f"{avg_metrics['f1']:.4f}    {result['trainable_params']:<12,} {result['gpu_memory']:<12.2f}")

    print("="*80)

# Example usage
if __name__ == "__main__":
    # Run the comparative study
    results = run_comparative_study()
    print_comparison_table(results)

    # Print detailed pathology-wise results for best model
    best_model_name = max(results.items(), key=lambda x: x[1]['best_auc'])[0]
    best_metrics = results[best_model_name]['metrics']

    print(f"\nBest Model: {best_model_name}")
    print("Pathology-wise Performance:")
    for pathology, metrics in best_metrics['per_pathology'].items():
        print(f"{pathology:<15} AUC: {metrics['auc']:.4f}, Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

Calculated classifier input size: 2560
Calculated classifier input size: 2560
Calculated classifier input size: 2560

=== Training Concatenation Fusion ===
Trainable parameters: 24,732,165
Estimated GPU memory: 112.72 MB
Epoch 1/20
Train Loss: 0.6936
Val AUC: 0.4109
Val Acc: 0.6000
Val F1: 0.4305
---
Epoch 2/20
Train Loss: 0.6923
Val AUC: 0.6186
Val Acc: 0.5600
Val F1: 0.5454
---
Epoch 3/20
Train Loss: 0.6925
Val AUC: 0.5299
Val Acc: 0.4500
Val F1: 0.4322
---
Epoch 4/20
Train Loss: 0.6946
Val AUC: 0.4333
Val Acc: 0.5700
Val F1: 0.5838
---
Epoch 5/20
Train Loss: 0.6933
Val AUC: 0.6545
Val Acc: 0.5700
Val F1: 0.5812
---
Epoch 6/20
Train Loss: 0.6929
Val AUC: 0.5120
Val Acc: 0.5000
Val F1: 0.5494
---
Epoch 7/20
Train Loss: 0.6920
Val AUC: 0.5214
Val Acc: 0.5000
Val F1: 0.5632
---
Epoch 8/20
Train Loss: 0.6931
Val AUC: 0.5625
Val Acc: 0.4800
Val F1: 0.5124
---
Epoch 9/20
Train Loss: 0.6935
Val AUC: 0.4835
Val Acc: 0.5100
Val F1: 0.5370
---
Epoch 10/20
Train Loss: 0.6949
Val AUC: 0.5468
Val